## 📑 Core Inflow Min Amounts
 
#### 🔗 Links
- **[BRD link](https://cepalhellas.atlassian.net/wiki/x/AYAPH)**  
  Comprehensive documentation outlining the business requirements for this report.

##### Authentication

In [0]:
import boto3
from botocore.exceptions import NoCredentialsError
import traceback

"""
Accessing the Hive metastore requires specific considerations:

1. Hive metastore access is not possible via the instance profile if the cluster is on shared access mode.
   The cluster must be in shared access mode to interact with the Unity catalog.

2. To access both Unity catalog and AWS services, an access token from Amazon is required.

3. To retrieve the token secret key, access the secrets scope 'aws-databricks,' where the access key is stored per environment.
    https://docs.databricks.com/en/security/secrets/index.html

4. Once AWS credentials are obtained, clients for services like S3 and SSM can be created.

5. After you authenticate successfully querying the Hive metastore is also working.
"""

try:
    mode = dbutils.widgets.get("mode")
except Exception as e:
    mode = "shared" 

# Function to get AWS credentials from Databricks secrets.
def get_aws_credentials(secrets_scope, key_name):
    try:
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None
    
# Function to get AWS credentials from Databricks secrets.
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration.
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
    
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

secrets_scope = "aws-databricks"
aws_keys = get_scope_key_names(secrets_scope)

aws_key = aws_keys[0] if aws_keys else None

# Get AWS secret key from Databricks secrets.
aws_secret_key = get_aws_credentials(secrets_scope,aws_key)

if mode == "shared":
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)

    S3_client = boto3.client('s3', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)                           
else:
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1')

    S3_client = boto3.client('s3', region_name='eu-central-1')                           


# Function to connect to AWS and access SSM parameter.
def connect_to_aws_and_access_ssm(ssm_key):

    if aws_secret_key is None:
        print("AWS secret key not available.")
        return

    # Access SSM parameter /cfn/a360/reports_output_path.
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )

    return response['Parameter']['Value']

# Get the Buckewt prefix from SSM to be sure.
env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix')

In [0]:
from pyspark.sql.functions import coalesce, concat_ws, lit, sha2, substring, when, col,regexp_replace

import delta_table_creator, extract, transform
from delta import DeltaTable  # type: ignore

# from pyspark.sql import SparkSession
# spark = SparkSession.builder.appName("example").getOrCreate()

import pandas as pd
from pyspark.sql import DataFrame
from pyspark.sql.functions import trim

import boto3
import os
 
#takes the values d for dev, qa for test and p for prod depending in which workspace the init variables script is running
# env = os.environ["S3PREFIX"]

dbutils.widgets.text("run_date", '')
run_date = dbutils.widgets.get("run_date").strip()

if not run_date:
    run_date = spark.sql(f'''
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.derivative.dexposure 
                         union
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.landed.profits_loan_details
                         ''').collect()

    if len(run_date) != 1: raise ValueError(f"Incorrect run_date!")
    run_date = run_date[0]['BusinessDate']
    run_date = run_date.strftime('%Y-%m-%d') 

##### Schema of the table

In [0]:
table_name = 'Core_InflowMinAmounts'

schema = (
"BusinessDate date,"
"ExposureKey string,"
"AssetSegment string,"
"CDI string,"
"SavedReason string,"
"FinalRiskSaved string,"
"FinalSaved string,"
"Bucket int,"
"AssignmentChannel string,"
"AssignmentChannelDetailed string,"
"AssignmentSegmentDetailed string,"
"AccountSegment string,"
"EBACategory string,"
"COA string,"
"`MinimumPayoutAmountb-1` decimal(19,2),"
"MinimumPayoutAmount decimal(19,2),"
"`AmountFor90-` decimal(19,2),"
"`MinAmount1+` decimal(19,2),"
"MinAmount decimal(19,2),"
"Flag string,"
"InsertTimeStamp timestamp"
)

In [0]:
%python
partition_columns = ["BusinessDate"]
delivery_table_path = ""
 
# ================== Create Delta Table ==================
delivery_table_path = f"`{env_prefix}-cdh-delivery`.repoint.{table_name}"
delta_table_creator.create_unity_delta_table(
    spark, delivery_table_path, partition_columns, schema
)
# if not DeltaTable.isDeltaTable(spark, delivery_table_path):
#     delta_table_creator.create_unity_delta_table(
#         spark, delivery_table_path, partition_columns, schema
#     )
  
# ================== Delete From Delta Table ==================
transform.delete_from_unity_table(
    spark,
    delivery_table_path,
    run_date,  # type: ignore
)


##### LPFXB248

In [0]:
LPFXB248 = spark.sql(
f'''
	SELECT	a.*
	FROM `{env_prefix}-cdh`.externalfiles.extlpfxb248 a
	INNER JOIN (SELECT	ExposureKeySIGLO, CDI, max(IngestionTimeStamp) IngestionTimeStamp
			    FROM `{env_prefix}-cdh`.externalfiles.extlpfxb248
      		    WHERE BusinessDate = {run_date!r}
				GROUP BY ExposureKeySIGLO, CDI) b ON a.CDI = b.CDI and a.ExposureKeySIGLO = b.ExposureKeySIGLO AND a.IngestionTimeStamp = b.IngestionTimeStamp
	WHERE a.BusinessDate = {run_date!r}
'''
)
LPFXB248.createOrReplaceTempView("LPFXB248")

# Check if the dataset is empty and print a message accordingly
if LPFXB248.count() == 0:
    print(f"The required external file extlpfxb248 dataset for date {run_date!r} is empty")

In [0]:
LPFXB248_Version2 = spark.sql(      
f'''
   SELECT BusinessDate
        , CDI 
        , PartyTaxIdentificationNumber 
        , ExposureKeySIGLO
        , CASE WHEN CAST(REPLACE(CapitalInArrears_1, ',', '0.') AS NUMERIC(19,2)) > 1
               THEN CAST(REPLACE(CapitalInArrears_1, ',', '.') AS NUMERIC(19,2))
               ELSE CAST(REPLACE(CapitalInArrears_1, ',', '0.') AS NUMERIC(19,2)) END AS CapitalInArrears_1
        , CASE WHEN CAST(REPLACE(CapitalInArrears_2, ',', '0.') AS NUMERIC(19,2)) > 1
               THEN CAST(REPLACE(CapitalInArrears_2, ',', '.') AS NUMERIC(19,2))
               ELSE CAST(REPLACE(CapitalInArrears_2, ',', '0.') AS NUMERIC(19,2)) END AS CapitalInArrears_2
        , CASE WHEN CAST(REPLACE(CapitalInArrears_3, ',', '0.') AS NUMERIC(19,2)) > 1
               THEN CAST(REPLACE(CapitalInArrears_3, ',', '.') AS NUMERIC(19,2))
               ELSE CAST(REPLACE(CapitalInArrears_3, ',', '0.') AS NUMERIC(19,2)) END AS CapitalInArrears_3
        , CASE WHEN CAST(REPLACE(CapitalInArrears_4, ',', '0.') AS NUMERIC(19,2)) > 1
               THEN CAST(REPLACE(CapitalInArrears_4, ',', '.') AS NUMERIC(19,2))
               ELSE CAST(REPLACE(CapitalInArrears_4, ',', '0.') AS NUMERIC(19,2)) END AS CapitalInArrears_4
        , CASE WHEN CAST(REPLACE(CapitalInArrears_5, ',', '0.') AS NUMERIC(19,2)) > 1
               THEN CAST(REPLACE(CapitalInArrears_5, ',', '.') AS NUMERIC(19,2))
               ELSE CAST(REPLACE(CapitalInArrears_5, ',', '0.') AS NUMERIC(19,2)) END AS CapitalInArrears_5
        , CASE WHEN CAST(REPLACE(CapitalInArrears_6, ',', '0.') AS NUMERIC(19,2)) > 1
               THEN CAST(REPLACE(CapitalInArrears_6, ',', '.') AS NUMERIC(19,2))
               ELSE CAST(REPLACE(CapitalInArrears_6, ',', '0.') AS NUMERIC(19,2)) END AS CapitalInArrears_6
        , CASE WHEN CAST(REPLACE(CapitalInArrears_7, ',', '0.') AS NUMERIC(19,2)) > 1
               THEN CAST(REPLACE(CapitalInArrears_7, ',', '.') AS NUMERIC(19,2))
               ELSE CAST(REPLACE(CapitalInArrears_7, ',', '0.') AS NUMERIC(19,2)) END AS CapitalInArrears_7
        , CASE WHEN CAST(REPLACE(CapitalInArrears_8, ',', '0.') AS NUMERIC(19,2)) > 1
               THEN CAST(REPLACE(CapitalInArrears_8, ',', '.') AS NUMERIC(19,2))
               ELSE CAST(REPLACE(CapitalInArrears_8, ',', '0.') AS NUMERIC(19,2)) END AS CapitalInArrears_8
        , CASE WHEN CAST(REPLACE(InterestInArrears_1, ',', '0.') AS NUMERIC(19,2)) > 1
               THEN CAST(REPLACE(InterestInArrears_1, ',', '.') AS NUMERIC(19,2))
               ELSE CAST(REPLACE(InterestInArrears_1, ',', '0.') AS NUMERIC(19,2)) END AS InterestInArrears_1
        , CASE WHEN CAST(REPLACE(InterestInArrears_2, ',', '0.') AS NUMERIC(19,2)) > 1
               THEN CAST(REPLACE(InterestInArrears_2, ',', '.') AS NUMERIC(19,2))
               ELSE CAST(REPLACE(InterestInArrears_2, ',', '0.') AS NUMERIC(19,2)) END AS InterestInArrears_2
        , CASE WHEN CAST(REPLACE(InterestInArrears_3, ',', '0.') AS NUMERIC(19,2)) > 1
               THEN CAST(REPLACE(InterestInArrears_3, ',', '.') AS NUMERIC(19,2))
               ELSE CAST(REPLACE(InterestInArrears_3, ',', '0.') AS NUMERIC(19,2)) END AS InterestInArrears_3
        , CASE WHEN CAST(REPLACE(InterestInArrears_4, ',', '0.') AS NUMERIC(19,2)) > 1
               THEN CAST(REPLACE(InterestInArrears_4, ',', '.') AS NUMERIC(19,2))
               ELSE CAST(REPLACE(InterestInArrears_4, ',', '0.') AS NUMERIC(19,2)) END AS InterestInArrears_4
        , CASE WHEN CAST(REPLACE(InterestInArrears_5, ',', '0.') AS NUMERIC(19,2)) > 1
               THEN CAST(REPLACE(InterestInArrears_5, ',', '.') AS NUMERIC(19,2))
               ELSE CAST(REPLACE(InterestInArrears_5, ',', '0.') AS NUMERIC(19,2)) END AS InterestInArrears_5
        , CASE WHEN CAST(REPLACE(InterestInArrears_6, ',', '0.') AS NUMERIC(19,2)) > 1
               THEN CAST(REPLACE(InterestInArrears_6, ',', '.') AS NUMERIC(19,2))
               ELSE CAST(REPLACE(InterestInArrears_6, ',', '0.') AS NUMERIC(19,2)) END AS InterestInArrears_6
        , CASE WHEN CAST(REPLACE(InterestInArrears_7, ',', '0.') AS NUMERIC(19,2)) > 1
               THEN CAST(REPLACE(InterestInArrears_7, ',', '.') AS NUMERIC(19,2))
               ELSE CAST(REPLACE(InterestInArrears_7, ',', '0.') AS NUMERIC(19,2)) END AS InterestInArrears_7
        , CASE WHEN CAST(REPLACE(InterestInArrears_8, ',', '0.') AS NUMERIC(19,2)) > 1
               THEN CAST(REPLACE(InterestInArrears_8, ',', '.') AS NUMERIC(19,2))
               ELSE CAST(REPLACE(InterestInArrears_8, ',', '0.') AS NUMERIC(19,2)) END AS InterestInArrears_8
        , CASE WHEN CAST(REPLACE(ExpenseInArrears_1, ',', '0.') AS NUMERIC(19,2)) > 1
               THEN CAST(REPLACE(ExpenseInArrears_1, ',', '.') AS NUMERIC(19,2))
               ELSE CAST(REPLACE(ExpenseInArrears_1, ',', '0.') AS NUMERIC(19,2)) END AS ExpenseInArrears_1
        , CASE WHEN CAST(REPLACE(ExpenseInArrears_2, ',', '0.') AS NUMERIC(19,2)) > 1
               THEN CAST(REPLACE(ExpenseInArrears_2, ',', '.') AS NUMERIC(19,2))
               ELSE CAST(REPLACE(ExpenseInArrears_2, ',', '0.') AS NUMERIC(19,2)) END AS ExpenseInArrears_2
        , CASE WHEN CAST(REPLACE(ExpenseInArrears_3, ',', '0.') AS NUMERIC(19,2)) > 1
               THEN CAST(REPLACE(ExpenseInArrears_3, ',', '.') AS NUMERIC(19,2))
               ELSE CAST(REPLACE(ExpenseInArrears_3, ',', '0.') AS NUMERIC(19,2)) END AS ExpenseInArrears_3
        , CASE WHEN CAST(REPLACE(ExpenseInArrears_4, ',', '0.') AS NUMERIC(19,2)) > 1
               THEN CAST(REPLACE(ExpenseInArrears_4, ',', '.') AS NUMERIC(19,2))
               ELSE CAST(REPLACE(ExpenseInArrears_4, ',', '0.') AS NUMERIC(19,2)) END AS ExpenseInArrears_4
        , CASE WHEN CAST(REPLACE(ExpenseInArrears_5, ',', '0.') AS NUMERIC(19,2)) > 1
               THEN CAST(REPLACE(ExpenseInArrears_5, ',', '.') AS NUMERIC(19,2))
               ELSE CAST(REPLACE(ExpenseInArrears_5, ',', '0.') AS NUMERIC(19,2)) END AS ExpenseInArrears_5
        , CASE WHEN CAST(REPLACE(ExpenseInArrears_6, ',', '0.') AS NUMERIC(19,2)) > 1
               THEN CAST(REPLACE(ExpenseInArrears_6, ',', '.') AS NUMERIC(19,2))
               ELSE CAST(REPLACE(ExpenseInArrears_6, ',', '0.') AS NUMERIC(19,2)) END AS ExpenseInArrears_6
        , CASE WHEN CAST(REPLACE(ExpenseInArrears_7, ',', '0.') AS NUMERIC(19,2)) > 1
               THEN CAST(REPLACE(ExpenseInArrears_7, ',', '.') AS NUMERIC(19,2))
               ELSE CAST(REPLACE(ExpenseInArrears_7, ',', '0.') AS NUMERIC(19,2)) END AS ExpenseInArrears_7
        , CASE WHEN CAST(REPLACE(ExpenseInArrears_8, ',', '0.') AS NUMERIC(19,2)) > 1
               THEN CAST(REPLACE(ExpenseInArrears_8, ',', '.') AS NUMERIC(19,2))
               ELSE CAST(REPLACE(ExpenseInArrears_8, ',', '0.') AS NUMERIC(19,2)) END AS ExpenseInArrears_8
        , CASE WHEN CAST(REPLACE(DueInterest, ',', '0.') AS NUMERIC(19,2)) > 1
               THEN CAST(REPLACE(DueInterest, ',', '.') AS NUMERIC(19,2))
               ELSE CAST(REPLACE(DueInterest, ',', '0.') AS NUMERIC(19,2)) END AS DueInterest
             
   FROM LPFXB248
'''
)
LPFXB248_Version2.createOrReplaceTempView("LPFXB248_Version2")

##### exposure

In [0]:
exposure = spark.sql(
f'''
	  SELECT	de.*
          , (SELECT max(fx.filled_fxrate)
             FROM  `{env_prefix}-bi`.common.fx_rates fx
             WHERE fx.OriginalCurrency = de.Currency
             AND fx.Date = de.BusinessDate) AS ExchangeRate
	  FROM `{env_prefix}-cdh`.primary.exposure de
	  WHERE BusinessDate = {run_date!r} 
      AND IsActiveFlag = TRUE
      AND PortfolioName = 'Core Retail' 
      AND PartyRelationshipType = 'Primary Owner'
'''
)
exposure.createOrReplaceTempView("exposure")

##### saved info

In [0]:
Saved = spark.sql(      
f'''
    SELECT *
    FROM `{env_prefix}-cdh`.derivative.dportfoliotoolcore
	WHERE BusinessDate = {run_date!r}  
'''
)
Saved.createOrReplaceTempView("Saved")

##### collection report

In [0]:
CollectionReport = spark.sql(      
f'''
    SELECT *
    FROM `{env_prefix}-cdh-delivery`.repoint.core_collectionsreport
	WHERE BusinessDate = {run_date!r}  
'''
)
CollectionReport.createOrReplaceTempView("CollectionReport")

##### modifications

In [0]:
Modifications = spark.sql(      
f'''
	SELECT	DISTINCT 
			M.OldExposureKey
		 ,	M.Categoryfinal
		 ,	M.ModificationDate
		 ,	RANK() OVER (PARTITION BY M.OldExposureKey ORDER BY M.ModificationDate DESC, M.Categoryfinal) AS RNKMods
	FROM `{env_prefix}-cdh-delivery`.repoint.core_modifications AS M    
	WHERE 1=1
	AND M.BusinessDate = {run_date!r} 
'''
)
Modifications.createOrReplaceTempView("Modifications")

##### perimeter

In [0]:
AccountPerimeter = spark.sql(      
f'''
    SELECT 
			M.ExposureKey				
    	,   M.CDI	
		,   M.AssetSegment
		,   M.AssignmentChannel   
		,   M.AssignmentTeam  
		,   M.AssignmentSegmentDetailed  
		,   M.EBAStatus                     
		,   M.RiskBucket            		     
		,   M.CourseOfAction            
	FROM `{env_prefix}-cdh-delivery`.repoint.core_assignmentdatamapping AS M
	WHERE 1=1
    AND M.BusinessDate = {run_date!r}
'''
)
AccountPerimeter.createOrReplaceTempView("AccountPerimeter")

In [0]:
AccountPerimeter_V2 = spark.sql(      
f'''
	SELECT M.Exposurekey		
	 	 , M.AssetSegment
    	 , M.CDI
    	 , ROUND(A.DueAmount/ (CASE WHEN A.ExchangeRate IS NULL THEN 1 ELSE A.ExchangeRate END),2)                   		 	AS DueAmountEur	
		 , ROUND(A.OutstandingTotalBalance / (CASE WHEN A.ExchangeRate IS NULL THEN 1 ELSE A.ExchangeRate END),2)     			AS OutstandingTotalBalanceEur 	
		 , M.AssignmentChannel
		 , M.AssignmentTeam
		 , M.AssignmentSegmentDetailed
		 , C.AssignDetail                                                                                              			AS AccountSegment    
		 , M.EBAStatus
		 , M.CourseOfAction
		 , A.Bucket                                                                                                    			AS Bucket
		 , M.RiskBucket
		 , ROUND(A.DueAmount/ (CASE WHEN A.ExchangeRate IS NULL THEN 1 ELSE A.ExchangeRate END) -100,2)                			AS `Rule100`
		 , ROUND(((100*A.DueAmount/ (CASE WHEN A.ExchangeRate IS NULL THEN 1 ELSE A.ExchangeRate END) )-
		  			 A.OutstandingTotalBalance/ (CASE WHEN A.ExchangeRate IS NULL THEN 1 ELSE A.ExchangeRate END) )/98.99 , 2)  AS `Rule1%`
      	 , CASE WHEN a.DueAmount IS NULL OR A.DueAmount > 0 THEN 1 ELSE 0 END 										  			AS `K1`
	FROM AccountPerimeter          M
	LEFT JOIN exposure              A ON A.ExposureKey = M.ExposureKey
	LEFT JOIN Modifications		Mods ON Mods.OldExposureKey = M.ExposureKey AND RNKMods = 1
	LEFT JOIN CollectionReport     C ON a.ExposureKey = c.ExposureKey
	WHERE 1=1
	AND (M.AssetSegment = 'SB' OR M.RiskBucket>= 1)
'''
)
AccountPerimeter_V2.createOrReplaceTempView("AccountPerimeter_V2")

##### Calculations

In [0]:
KTE = spark.sql(      
f'''
	SELECT *
		 , CASE WHEN (IFNULL(CapitalInArrears_1,0))  > 0 THEN 1 ELSE 0 END AS K1
		 , CASE WHEN (IFNULL(CapitalInArrears_2,0))  > 0 THEN 2 ELSE 0 END AS K2
		 , CASE WHEN (IFNULL(CapitalInArrears_3,0))  > 0 THEN 3 ELSE 0 END AS K3
		 , CASE WHEN (IFNULL(CapitalInArrears_4,0))  > 0 THEN 4 ELSE 0 END AS K4
         , CASE WHEN (IFNULL(CapitalInArrears_5,0) + IFNULL(CapitalInArrears_6,0) + IFNULL(CapitalInArrears_7,0) + IFNULL(CapitalInArrears_8,0)) > 0 THEN 5 ELSE 0 END AS K5
		 , CASE WHEN (IFNULL(InterestInArrears_1,0))  > 0 THEN 1 ELSE 0 END AS T1
		 , CASE WHEN (IFNULL(InterestInArrears_2,0))  > 0 THEN 2 ELSE 0 END AS T2
		 , CASE WHEN (IFNULL(InterestInArrears_3,0))  > 0 THEN 3 ELSE 0 END AS T3
		 , CASE WHEN (IFNULL(InterestInArrears_4,0))  > 0 THEN 4 ELSE 0 END AS T4
         , CASE WHEN (IFNULL(InterestInArrears_5,0) + IFNULL(InterestInArrears_6,0) + IFNULL(InterestInArrears_7,0) + IFNULL(InterestInArrears_8,0)) > 0 THEN 5 ELSE 0 END AS T5
		 , CASE WHEN (IFNULL(ExpenseInArrears_1,0))  > 0 THEN 1 ELSE 0 END AS E1
		 , CASE WHEN (IFNULL(ExpenseInArrears_2,0))  > 0 THEN 2 ELSE 0 END AS E2
		 , CASE WHEN (IFNULL(ExpenseInArrears_3,0))  > 0 THEN 3 ELSE 0 END AS E3
		 , CASE WHEN (IFNULL(ExpenseInArrears_4,0))  > 0 THEN 4 ELSE 0 END AS E4
         , CASE WHEN (IFNULL(ExpenseInArrears_5,0) + IFNULL(ExpenseInArrears_6,0) + IFNULL(ExpenseInArrears_7,0) + IFNULL(ExpenseInArrears_8,0)) > 0 THEN 5 ELSE 0 END AS E5
	FROM LPFXB248_Version2
'''
)
KTE.createOrReplaceTempView("KTE")

In [0]:
E1MAX = spark.sql(      
f'''
        SELECT `ExposureKeySIGLO`, `1`, `2`, `3`, `4`, `5`
        FROM KTE
        PIVOT  
        ( 
        SUM(E1) as E1Sum
          FOR E1 IN ('1' AS `1`)       
        )
        PIVOT  
        (  
        SUM(E2) as E2Sum
          FOR E2 IN ('2' AS `2`)    
        ) 
        PIVOT  
        (  
        SUM(E3) as E3Sum
          FOR E3 IN ('3' AS `3`)    
        ) 
        PIVOT  
        (  
        SUM(E4) as E4Sum
          FOR E4 IN ('4' AS `4`)    
        ) 
        PIVOT  
        (  
        SUM(E5) as E5Sum
          FOR E5 IN ('5' AS `5`)    
        ) 
'''
)
E1MAX.createOrReplaceTempView("E1MAX")

In [0]:
EMAX = spark.sql(      
f'''
		SELECT `ExposureKeySIGLO`
		      , E     
          , RANK() OVER (PARTITION BY `ExposureKeySIGLO` ORDER BY E DESC) AS E_RNKMods
    FROM  
        (
          SELECT `ExposureKeySIGLO`
                ,`1`
                ,`2`
                ,`3`
                ,`4`
        		    ,`5`     
          FROM E1MAX
        ) AS SourceTable  
        UNPIVOT  
        ( AccountNumber
          for E in (`1`,`2`,`3`,`4`,`5`)   
        ) AS P1
'''
)
EMAX.createOrReplaceTempView("EMAX")

In [0]:
K1MAX = spark.sql(      
f'''
    SELECT `ExposureKeySIGLO`, `1`, `2`, `3`, `4`, `5`
    FROM  KTE
        PIVOT  
        (  
          SUM(K1) AS K1Sum
          FOR K1 IN ('1' AS `1`) 
        ) 
        
        PIVOT  
        (  
          SUM(K2) AS K2Sum
          FOR K2 IN ('2' AS `2`) 
        ) 
        
        PIVOT  
        (  
          SUM(K3) AS K3Sum
          FOR K3 IN ('3' AS `3`) 
        ) 
        
        PIVOT  
        (  
          SUM(K4) AS K4Sum
          FOR K4 IN ('4' AS`4`) 
        )
        
        PIVOT  
        (  
          SUM(K5) AS K5Sum
          FOR K5 IN ('5' AS `5`) 
        ) 
'''
)
K1MAX.createOrReplaceTempView("K1MAX")

In [0]:
KMAX = spark.sql(      
f'''
		SELECT `ExposureKeySIGLO`
		     , K     
         , RANK() OVER (PARTITION BY `ExposureKeySIGLO` ORDER BY K DESC) AS K_RNKMods
    FROM  
        (
          SELECT `ExposureKeySIGLO`
                ,`1`
                ,`2`
                ,`3`
                ,`4`
        		,`5`      
          FROM K1MAX
        ) AS SourceTable  
        UNPIVOT  
        ( AccountNumber
          for K in (`1`,`2`,`3`,`4`,`5`)   
        ) AS P1
'''
)
KMAX.createOrReplaceTempView("KMAX")

In [0]:
T1MAX = spark.sql(      
f'''
    SELECT `ExposureKeySIGLO`, `1`,`2`,`3`,`4`,`5`
    FROM KTE

        PIVOT  
        (  
          SUM(T1) AS T1Sum
          FOR T1 IN ('1' AS `1`) 
        )
        
        PIVOT  
        (  
          SUM(T2) AS T2Sum
          FOR T2 IN ('2' AS `2`) 
        )

        PIVOT  
        (  
          SUM(T3) AS T3Sum
          FOR T3 IN ('3' AS `3`) 
        )
        
        PIVOT  
        (  
          SUM(T4) AS T4Sum
          FOR T4 IN ('4' AS `4`) 
        )
        
        PIVOT  
        (  
          SUM(T5) AS T5Sum
          FOR T5 IN ('5' AS `5`) 
        )
'''
)
T1MAX.createOrReplaceTempView("T1MAX")

In [0]:
TMAX = spark.sql(      
f'''
		SELECT `ExposureKeySIGLO`
		     , T     
             , RANK() OVER (PARTITION BY `ExposureKeySIGLO` ORDER BY T DESC) AS T_RNKMods
    FROM  
        (
          SELECT `ExposureKeySIGLO`
                ,`1`
                ,`2`
                ,`3`
                ,`4`
        		,`5`      
          FROM T1MAX
        ) AS SourceTable  
        UNPIVOT  
        ( AccountNumber
          for T in (`1`,`2`,`3`,`4`,`5`)   
        ) AS P1
'''
)
TMAX.createOrReplaceTempView("TMAX")

In [0]:
MAX_E_K_T = spark.sql(      
f'''
    SELECT L.`ExposureKeySIGLO`
	     , E
		 , T
		 , K
		 , CASE WHEN E_RNKMods = 1 THEN (CASE WHEN E IS NULL THEN 0 ELSE E END) END AS `EMax`
		 , CASE WHEN E_RNKMods = 2 THEN (CASE WHEN E IS NULL THEN 0 ELSE E END) END AS `EMax-1`
		 , CASE WHEN K_RNKMods = 1 THEN (CASE WHEN K IS NULL THEN 0 ELSE K END) END AS `KMax`
		 , CASE WHEN K_RNKMods = 2 THEN (CASE WHEN K IS NULL THEN 0 ELSE K END) END AS `KMax-1`
		 , CASE WHEN T_RNKMods = 1 THEN (CASE WHEN T IS NULL THEN 0 ELSE T END) END AS `TMax`
		 , CASE WHEN T_RNKMods = 2 THEN (CASE WHEN T IS NULL THEN 0 ELSE T END) END AS `TMax-1`           
	FROM LPFXB248_Version2  L
	LEFT JOIN EMAX E
	ON L.`ExposureKeySIGLO` = E.`ExposureKeySIGLO`
	LEFT JOIN TMAX T
	ON L.`ExposureKeySIGLO` = T.`ExposureKeySIGLO`
	LEFT JOIN KMAX K
	ON L.`ExposureKeySIGLO` = K.`ExposureKeySIGLO`
'''
)
MAX_E_K_T.createOrReplaceTempView("MAX_E_K_T")

In [0]:
MAX1 = spark.sql(      
f'''
	SELECT K.`ExposureKeySIGLO` 
	     , MAX(M1.`EMax-1`) AS `EMax-1`
		 , MAX(M1.`KMax-1`) AS `KMax-1`
		 , MAX(M1.`TMax-1`) AS `TMax-1`
	     , MAX(M1.KMax)   AS MaxBucketCapital
	     , MAX(M1.TMax)   AS MaxBucketInterests
	     , MAX(M1.EMax)   AS MaxBucketExpenses
		 , CASE WHEN MAX(M1.EMax) > MAX(M1.TMax) THEN MAX(M1.EMax)
		        WHEN MAX(M1.EMax) < MAX(M1.TMax) THEN MAX(M1.TMax)
				ELSE MAX(M1.TMax) END AS MaxInterestsExpenses
	FROM LPFXB248_Version2         K    
	LEFT JOIN MAX_E_K_T  M1
	ON M1.`ExposureKeySIGLO` = K.`ExposureKeySIGLO`
	GROUP BY K.`ExposureKeySIGLO`
'''
)
MAX1.createOrReplaceTempView("MAX1")

In [0]:
MAX1_1 = spark.sql(      
f'''
	SELECT  K.*
		  , CASE WHEN (CASE WHEN `MaxInterestsExpenses` IS NULL THEN 0 ELSE `MaxInterestsExpenses` END) <> 
                      (CASE WHEN `MaxBucketCapital` IS NULL THEN 0 ELSE `MaxBucketCapital` END) THEN 
                      (CASE WHEN `MaxBucketCapital` IS NULL THEN 0 ELSE `MaxBucketCapital` END)
	        ELSE (CASE WHEN `KMax-1` IS NULL THEN 0 ELSE `KMax-1` END) 
            END AS `MaxBucketCapital-1`
	      , CASE WHEN (CASE WHEN `MaxBucketExpenses` IS NULL THEN 0 ELSE `MaxBucketExpenses` END) > 
                      (CASE WHEN `MaxBucketInterests` IS NULL THEN 0 ELSE `MaxBucketInterests` END) AND 
                      (CASE WHEN `MaxBucketExpenses` IS NULL THEN 0 ELSE `MaxBucketExpenses` END) > 
                      (CASE WHEN `MaxBucketCapital` IS NULL THEN 0 ELSE `MaxBucketCapital` END) THEN 
                      (CASE WHEN `MaxBucketInterests` IS NULL THEN 0 ELSE `MaxBucketInterests` END)
		         WHEN (CASE WHEN `MaxBucketCapital` IS NULL THEN 0 ELSE `MaxBucketCapital` END) >= 
                      (CASE WHEN `MaxBucketInterests` IS NULL THEN 0 ELSE `MaxBucketInterests` END) THEN 0
			ELSE (CASE WHEN `TMax-1` IS NULL THEN 0 ELSE `TMax-1` END) 
            END AS `MaxBucketInterests-1`
		  , CASE WHEN (CASE WHEN `MaxBucketInterests` IS NULL THEN 0 ELSE `MaxBucketInterests` END) >= 
                      (CASE WHEN `MaxBucketExpenses` IS NULL THEN 0 ELSE `MaxBucketExpenses` END) OR 
                      (CASE WHEN `MaxBucketCapital` IS NULL THEN 0 ELSE `MaxBucketCapital` END) >= 
                      (CASE WHEN `MaxBucketExpenses` IS NULL THEN 0 ELSE `MaxBucketExpenses` END) THEN 0
		    ELSE (CASE WHEN `EMax-1` IS NULL THEN 0 ELSE `EMax-1` END) 
            END AS `MaxBucketExpenses-1`
	FROM MAX1  K
'''
)
MAX1_1.createOrReplaceTempView("MAX1_1")

In [0]:
Max2 = spark.sql(      
f'''
	SELECT l.*
	     , (CASE WHEN MaxBucketCapital IS NULL THEN 0 ELSE MaxBucketCapital END)                   AS `MaxBucketCapital`
		, (CASE WHEN MaxBucketInterests IS NULL THEN 0 ELSE MaxBucketInterests END)               AS `MaxBucketInterests`
		, (CASE WHEN MaxBucketExpenses IS NULL THEN 0 ELSE MaxBucketExpenses END)                 AS `MaxBucketExpenses`
		, (CASE WHEN `MaxBucketCapital-1` IS NULL THEN 0 ELSE `MaxBucketCapital-1` END)           AS `MaxBucketCapital-1`
		, (CASE WHEN `MaxBucketInterests-1` IS NULL THEN 0 ELSE `MaxBucketInterests-1` END)       AS `MaxBucketInterests-1`
		, (CASE WHEN `MaxBucketExpenses-1` IS NULL THEN 0 ELSE `MaxBucketExpenses-1` END)        AS `MaxBucketExpenses-1`
		, CASE WHEN (CASE WHEN MaxBucketCapital IS NULL THEN 0 ELSE MaxBucketCapital END) = 0 THEN 0
		       WHEN (CASE WHEN MaxBucketCapital IS NULL THEN 0 ELSE MaxBucketCapital END) = 1 THEN 
                      (CASE WHEN `CapitalInArrears_1` IS NULL THEN 0 ELSE `CapitalInArrears_1` END)
		 	  WHEN (CASE WHEN MaxBucketCapital IS NULL THEN 0 ELSE MaxBucketCapital END) = 2 THEN 
                      (CASE WHEN `CapitalInArrears_2` IS NULL THEN 0 ELSE `CapitalInArrears_2` END)
			  WHEN (CASE WHEN MaxBucketCapital IS NULL THEN 0 ELSE MaxBucketCapital END) = 3 THEN 
                      (CASE WHEN `CapitalInArrears_3` IS NULL THEN 0 ELSE `CapitalInArrears_3` END)
			  WHEN (CASE WHEN MaxBucketCapital IS NULL THEN 0 ELSE MaxBucketCapital END) = 4 THEN 
                      (CASE WHEN `CapitalInArrears_4` IS NULL THEN 0 ELSE `CapitalInArrears_4` END)
			  WHEN (CASE WHEN MaxBucketCapital IS NULL THEN 0 ELSE MaxBucketCapital END) = 5 THEN 
                      (CASE WHEN `CapitalInArrears_5` IS NULL THEN 0 ELSE `CapitalInArrears_5` END) +
			       (CASE WHEN `CapitalInArrears_6` IS NULL THEN 0 ELSE `CapitalInArrears_6` END) +
                      (CASE WHEN `CapitalInArrears_7` IS NULL THEN 0 ELSE `CapitalInArrears_7` END) +
                      (CASE WHEN `CapitalInArrears_8` IS NULL THEN 0 ELSE `CapitalInArrears_8` END)
		  ELSE '9999999999' 
            END AS MaxBucketCapitalAmount
			 , CASE WHEN (CASE WHEN MaxBucketInterests IS NULL THEN 0 ELSE MaxBucketInterests END) = 0 THEN 0
			        WHEN (CASE WHEN MaxBucketInterests IS NULL THEN 0 ELSE MaxBucketInterests END) = 1 THEN 
                         (CASE WHEN `InterestInArrears_1` IS NULL THEN 0 ELSE `InterestInArrears_1` END)
					WHEN (CASE WHEN MaxBucketInterests IS NULL THEN 0 ELSE MaxBucketInterests END) = 2 THEN 
                         (CASE WHEN `InterestInArrears_2` IS NULL THEN 0 ELSE `InterestInArrears_2` END)
					WHEN (CASE WHEN MaxBucketInterests IS NULL THEN 0 ELSE MaxBucketInterests END) = 3 THEN 
                         (CASE WHEN `InterestInArrears_3` IS NULL THEN 0 ELSE `InterestInArrears_3` END)
					WHEN (CASE WHEN MaxBucketInterests IS NULL THEN 0 ELSE MaxBucketInterests END) = 4 THEN 
                         (CASE WHEN `InterestInArrears_4` IS NULL THEN 0 ELSE `InterestInArrears_4` END)
					WHEN (CASE WHEN MaxBucketInterests IS NULL THEN 0 ELSE MaxBucketInterests END) = 5 THEN 
                              (CASE WHEN `InterestInArrears_5` IS NULL THEN 0 ELSE `InterestInArrears_5` END) +
                              (CASE WHEN `InterestInArrears_6` IS NULL THEN 0 ELSE `InterestInArrears_6` END) +
                              (CASE WHEN `InterestInArrears_7` IS NULL THEN 0 ELSE `InterestInArrears_7` END) +
                              (CASE WHEN `InterestInArrears_8` IS NULL THEN 0 ELSE `InterestInArrears_8` END)
			   ELSE '9999999999' 
               END AS MaxBucketInterestsAmount
			 , CASE WHEN (CASE WHEN MaxBucketExpenses IS NULL THEN 0 ELSE MaxBucketExpenses END) = 0 THEN 0
			        WHEN (CASE WHEN MaxBucketExpenses IS NULL THEN 0 ELSE MaxBucketExpenses END) = 1 THEN 
                         (CASE WHEN `ExpenseInArrears_1` IS NULL THEN 0 ELSE `ExpenseInArrears_1` END)
					WHEN (CASE WHEN MaxBucketExpenses IS NULL THEN 0 ELSE MaxBucketExpenses END) = 2 THEN 
                         (CASE WHEN `ExpenseInArrears_2` IS NULL THEN 0 ELSE `ExpenseInArrears_2` END)
					WHEN (CASE WHEN MaxBucketExpenses IS NULL THEN 0 ELSE MaxBucketExpenses END) = 3 THEN 
                         (CASE WHEN `ExpenseInArrears_3` IS NULL THEN 0 ELSE `ExpenseInArrears_3` END)
					WHEN (CASE WHEN MaxBucketExpenses IS NULL THEN 0 ELSE MaxBucketExpenses END) = 4 THEN 
                         (CASE WHEN `ExpenseInArrears_4` IS NULL THEN 0 ELSE `ExpenseInArrears_4` END)
					WHEN (CASE WHEN MaxBucketExpenses IS NULL THEN 0 ELSE MaxBucketExpenses END) = 5 THEN 
                              (CASE WHEN `ExpenseInArrears_5` IS NULL THEN 0 ELSE `ExpenseInArrears_5` END) + 
                              (CASE WHEN `ExpenseInArrears_6` IS NULL THEN 0 ELSE `ExpenseInArrears_6` END) +
                              (CASE WHEN `ExpenseInArrears_7` IS NULL THEN 0 ELSE `ExpenseInArrears_7` END) +
                              (CASE WHEN `ExpenseInArrears_8` IS NULL THEN 0 ELSE `ExpenseInArrears_8` END)
			   ELSE '9999999999' 
               END AS MaxBucketExpensesAmount
             , (CASE WHEN `ExpenseInArrears_1` IS NULL THEN 0 ELSE `ExpenseInArrears_1` END) +
               (CASE WHEN `ExpenseInArrears_2` IS NULL THEN 0 ELSE `ExpenseInArrears_2` END) +
               (CASE WHEN `ExpenseInArrears_3` IS NULL THEN 0 ELSE `ExpenseInArrears_3` END) +
               (CASE WHEN `ExpenseInArrears_4` IS NULL THEN 0 ELSE `ExpenseInArrears_4` END) +
               (CASE WHEN `ExpenseInArrears_5` IS NULL THEN 0 ELSE `ExpenseInArrears_5` END) +
               (CASE WHEN `ExpenseInArrears_6` IS NULL THEN 0 ELSE `ExpenseInArrears_6` END) +
               (CASE WHEN `ExpenseInArrears_7` IS NULL THEN 0 ELSE `ExpenseInArrears_7` END) +
               (CASE WHEN `ExpenseInArrears_8` IS NULL THEN 0 ELSE `ExpenseInArrears_8` END) AS TotalExpenses
		   , (CASE WHEN `InterestInArrears_1` IS NULL THEN 0 ELSE `InterestInArrears_1` END) +
               (CASE WHEN `InterestInArrears_2` IS NULL THEN 0 ELSE `InterestInArrears_2` END) +
               (CASE WHEN `InterestInArrears_3` IS NULL THEN 0 ELSE `InterestInArrears_3` END) +
               (CASE WHEN `InterestInArrears_4` IS NULL THEN 0 ELSE `InterestInArrears_4` END) +
               (CASE WHEN `InterestInArrears_5` IS NULL THEN 0 ELSE `InterestInArrears_5` END) +
               (CASE WHEN `InterestInArrears_6` IS NULL THEN 0 ELSE `InterestInArrears_6` END) +
               (CASE WHEN `InterestInArrears_7` IS NULL THEN 0 ELSE `InterestInArrears_7` END) +
               (CASE WHEN `InterestInArrears_8` IS NULL THEN 0 ELSE `InterestInArrears_8` END) AS TotalInterests
		FROM LPFXB248_Version2     l
		LEFT JOIN MAX1_1  m
		ON l.`ExposureKeySIGLO` = m.`ExposureKeySIGLO`
'''
)
Max2.createOrReplaceTempView("Max2")

In [0]:
Max2_2 = spark.sql(      
f'''
		SELECT l.*
		     , CASE WHEN `MaxBucketCapital-1` = 0 THEN 0
			        WHEN `MaxBucketCapital-1` = 1 THEN (CASE WHEN `CapitalInArrears_1` IS NULL THEN 0 ELSE `CapitalInArrears_1` END)
					WHEN `MaxBucketCapital-1` = 2 THEN (CASE WHEN `CapitalInArrears_2` IS NULL THEN 0 ELSE `CapitalInArrears_2` END)
					WHEN `MaxBucketCapital-1` = 3 THEN (CASE WHEN `CapitalInArrears_3` IS NULL THEN 0 ELSE `CapitalInArrears_3` END)
					WHEN `MaxBucketCapital-1` = 4 THEN (CASE WHEN `CapitalInArrears_4` IS NULL THEN 0 ELSE `CapitalInArrears_4` END)
					WHEN `MaxBucketCapital-1` = 5 THEN (CASE WHEN `CapitalInArrears_5` IS NULL THEN 0 ELSE `CapitalInArrears_5` END) +
                                                         (CASE WHEN `CapitalInArrears_6` IS NULL THEN 0 ELSE `CapitalInArrears_6` END) +
                                                         (CASE WHEN `CapitalInArrears_7` IS NULL THEN 0 ELSE `CapitalInArrears_7` END) +
                                                         (CASE WHEN `CapitalInArrears_8` IS NULL THEN 0 ELSE `CapitalInArrears_8` END)
			   ELSE '9999999999' 
               END AS `MaxBucketCapitalAmount-1`
			 , CASE WHEN `MaxBucketInterests-1` = 0 THEN 0
			        WHEN `MaxBucketInterests-1` = 1 THEN (CASE WHEN `InterestInArrears_1` IS NULL THEN 0 ELSE `InterestInArrears_1` END)
					WHEN `MaxBucketInterests-1` = 2 THEN (CASE WHEN `InterestInArrears_2` IS NULL THEN 0 ELSE `InterestInArrears_2` END)
					WHEN `MaxBucketInterests-1` = 3 THEN (CASE WHEN `InterestInArrears_3` IS NULL THEN 0 ELSE `InterestInArrears_3` END)
					WHEN `MaxBucketInterests-1` = 4 THEN (CASE WHEN `InterestInArrears_4` IS NULL THEN 0 ELSE `InterestInArrears_4` END)
					WHEN `MaxBucketInterests-1` = 5 THEN (CASE WHEN `InterestInArrears_5` IS NULL THEN 0 ELSE `InterestInArrears_5` END) + 
                                                     (CASE WHEN `InterestInArrears_6` IS NULL THEN 0 ELSE `InterestInArrears_6` END) +
                                                     (CASE WHEN `InterestInArrears_7` IS NULL THEN 0 ELSE `InterestInArrears_7` END) +
                                                     (CASE WHEN `InterestInArrears_8` IS NULL THEN 0 ELSE `InterestInArrears_8` END)
		       ELSE '9999999999' 
               END AS `MaxBucketInterestsAmount-1`
			 , CASE WHEN `MaxBucketExpenses-1` = 0 THEN 0
			        WHEN `MaxBucketExpenses-1` = 1 THEN (CASE WHEN `ExpenseInArrears_1` IS NULL THEN 0 ELSE `ExpenseInArrears_1` END)
					WHEN `MaxBucketExpenses-1` = 2 THEN (CASE WHEN `ExpenseInArrears_2` IS NULL THEN 0 ELSE `ExpenseInArrears_2` END)
					WHEN `MaxBucketExpenses-1` = 3 THEN (CASE WHEN `ExpenseInArrears_3` IS NULL THEN 0 ELSE `ExpenseInArrears_3` END)
					WHEN `MaxBucketExpenses-1` = 4 THEN (CASE WHEN `ExpenseInArrears_4` IS NULL THEN 0 ELSE `ExpenseInArrears_4` END)
					WHEN `MaxBucketExpenses-1` = 5 THEN (CASE WHEN `ExpenseInArrears_5` IS NULL THEN 0 ELSE `ExpenseInArrears_5` END) + 
                                                       (CASE WHEN `ExpenseInArrears_6` IS NULL THEN 0 ELSE `ExpenseInArrears_6` END) +
                                                       (CASE WHEN `ExpenseInArrears_7` IS NULL THEN 0 ELSE `ExpenseInArrears_7` END) +
                                                       (CASE WHEN `ExpenseInArrears_8` IS NULL THEN 0 ELSE `ExpenseInArrears_8` END)
			   ELSE '9999999999' 
               END AS `MaxBucketExpensesAmount-1`
		FROM MAX2   l
'''
)
Max2_2.createOrReplaceTempView("Max2_2")

In [0]:
Max3 = spark.sql(      
f'''
		SELECT *
		     , (CASE WHEN TotalInterests IS NULL THEN 0 ELSE TotalInterests END) +
                 (CASE WHEN `DueInterest` IS NULL THEN 0 ELSE `DueInterest` END) AS TotalInterestsAndMLT
			, (CASE WHEN `CapitalInArrears_3` IS NULL THEN 0 ELSE `CapitalInArrears_3` END) + 
                 (CASE WHEN `CapitalInArrears_4` IS NULL THEN 0 ELSE `CapitalInArrears_4` END) +
                 (CASE WHEN `CapitalInArrears_5` IS NULL THEN 0 ELSE `CapitalInArrears_5` END) +
                 (CASE WHEN `CapitalInArrears_6` IS NULL THEN 0 ELSE `CapitalInArrears_6` END) +
                 (CASE WHEN `CapitalInArrears_7` IS NULL THEN 0 ELSE `CapitalInArrears_7` END) +
                 (CASE WHEN `CapitalInArrears_8` IS NULL THEN 0 ELSE `CapitalInArrears_8` END) AS `Capital>2`
			, (CASE WHEN `CapitalInArrears_1` IS NULL THEN 0 ELSE `CapitalInArrears_1` END) + 
                 (CASE WHEN `CapitalInArrears_2` IS NULL THEN 0 ELSE `CapitalInArrears_2` END) AS `Capital1-2`
			, (CASE WHEN `InterestInArrears_3` IS NULL THEN 0 ELSE `InterestInArrears_3` END) +
                 (CASE WHEN `InterestInArrears_4` IS NULL THEN 0 ELSE `InterestInArrears_4` END) +
                 (CASE WHEN `InterestInArrears_5` IS NULL THEN 0 ELSE `InterestInArrears_5` END) +
                 (CASE WHEN `InterestInArrears_6` IS NULL THEN 0 ELSE `InterestInArrears_6` END) +
                 (CASE WHEN `InterestInArrears_7` IS NULL THEN 0 ELSE `InterestInArrears_7` END) +
                 (CASE WHEN `InterestInArrears_8` IS NULL THEN 0 ELSE `InterestInArrears_8` END) +
                 (CASE WHEN `DueInterest` IS NULL THEN 0 ELSE `DueInterest` END) AS `MLTandInterests>2`
			, (CASE WHEN `InterestInArrears_1` IS NULL THEN 0 ELSE `InterestInArrears_1` END) +
                 (CASE WHEN `InterestInArrears_2` IS NULL THEN 0 ELSE `InterestInArrears_2` END) AS `Interests1-2`
			, (CASE WHEN `ExpenseInArrears_3` IS NULL THEN 0 ELSE `ExpenseInArrears_3` END) + 
                 (CASE WHEN `ExpenseInArrears_4` IS NULL THEN 0 ELSE `ExpenseInArrears_4` END) +
                 (CASE WHEN `ExpenseInArrears_5` IS NULL THEN 0 ELSE `ExpenseInArrears_5` END) +
			  (CASE WHEN `ExpenseInArrears_6` IS NULL THEN 0 ELSE `ExpenseInArrears_6` END) +
                 (CASE WHEN `ExpenseInArrears_7` IS NULL THEN 0 ELSE `ExpenseInArrears_7` END) +
                 (CASE WHEN `ExpenseInArrears_8` IS NULL THEN 0 ELSE `ExpenseInArrears_8` END) AS `Expenses>2`
			, (CASE WHEN `ExpenseInArrears_1` IS NULL THEN 0 ELSE `ExpenseInArrears_1` END) +
                 (CASE WHEN `ExpenseInArrears_2` IS NULL THEN 0 ELSE `ExpenseInArrears_2` END) AS `Expenses1-2`
			, (CASE WHEN MaxBucketInterestsAmount IS NULL THEN 0 ELSE MaxBucketInterestsAmount END) + 
                 (CASE WHEN `DueInterest` IS NULL THEN 0 ELSE `DueInterest` END) AS MaxBucketInterestsAndMLT
		FROM MAX2_2
'''
)
Max3.createOrReplaceTempView("Max3")

In [0]:
Max4 = spark.sql(      
f'''
		SELECT *
		     , CASE WHEN (CASE WHEN MaxBucketExpenses IS NULL THEN 0 ELSE MaxBucketExpenses END) > 
                         (CASE WHEN MaxBucketCapital IS NULL THEN 0 ELSE MaxBucketCapital END) AND 
                         (CASE WHEN MaxBucketExpenses IS NULL THEN 0 ELSE MaxBucketExpenses END) > 
                         (CASE WHEN MaxBucketInterests IS NULL THEN 0 ELSE MaxBucketInterests END) THEN 
                         (CASE WHEN MaxBucketExpenses IS NULL THEN 0 ELSE MaxBucketExpenses END)
			        WHEN (CASE WHEN MaxBucketCapital IS NULL THEN 0 ELSE MaxBucketCapital END) > 
                         (CASE WHEN MaxBucketExpenses IS NULL THEN 0 ELSE MaxBucketExpenses END) AND 
                         (CASE WHEN MaxBucketCapital IS NULL THEN 0 ELSE MaxBucketCapital END) > 
                         (CASE WHEN MaxBucketInterests IS NULL THEN 0 ELSE MaxBucketInterests END) THEN 
                         (CASE WHEN MaxBucketCapital IS NULL THEN 0 ELSE MaxBucketCapital END)
					WHEN (CASE WHEN MaxBucketInterests IS NULL THEN 0 ELSE MaxBucketInterests END) > 
                         (CASE WHEN MaxBucketCapital IS NULL THEN 0 ELSE MaxBucketCapital END) AND 
                         (CASE WHEN MaxBucketInterests IS NULL THEN 0 ELSE MaxBucketInterests END) > 
                         (CASE WHEN MaxBucketExpenses IS NULL THEN 0 ELSE MaxBucketExpenses END) THEN 
                         (CASE WHEN MaxBucketInterests IS NULL THEN 0 ELSE MaxBucketInterests END)
			  ELSE (CASE WHEN MaxBucketCapital IS NULL THEN 0 ELSE MaxBucketCapital END) 
              END AS MAX
            , CASE WHEN (CASE WHEN `MaxBucketExpenses-1` IS NULL THEN 0 ELSE `MaxBucketExpenses-1` END) > 
                        (CASE WHEN `MaxBucketCapital-1` IS NULL THEN 0 ELSE `MaxBucketCapital-1` END) AND 
                        (CASE WHEN `MaxBucketExpenses-1` IS NULL THEN 0 ELSE `MaxBucketExpenses-1` END) > 
                        (CASE WHEN `MaxBucketInterests-1` IS NULL THEN 0 ELSE `MaxBucketInterests-1` END) THEN 
                        (CASE WHEN `MaxBucketExpenses-1` IS NULL THEN 0 ELSE `MaxBucketExpenses-1` END)
			       WHEN (CASE WHEN `MaxBucketCapital-1` IS NULL THEN 0 ELSE `MaxBucketCapital-1` END) > 
                        (CASE WHEN `MaxBucketExpenses-1` IS NULL THEN 0 ELSE `MaxBucketExpenses-1` END) AND 
                        (CASE WHEN `MaxBucketCapital-1` IS NULL THEN 0 ELSE `MaxBucketCapital-1`END) > 
                        (CASE WHEN `MaxBucketInterests-1` IS NULL THEN 0 ELSE `MaxBucketInterests-1` END) THEN 
                        (CASE WHEN `MaxBucketCapital-1` IS NULL THEN 0 ELSE `MaxBucketCapital-1` END)
				   WHEN (CASE WHEN `MaxBucketInterests-1` IS NULL THEN 0 ELSE `MaxBucketInterests-1` END) > 
                        (CASE WHEN `MaxBucketCapital-1` IS NULL THEN 0 ELSE `MaxBucketCapital-1`END) AND 
                        (CASE WHEN `MaxBucketInterests-1` IS NULL THEN 0 ELSE `MaxBucketInterests-1` END) > 
                        (CASE WHEN `MaxBucketExpenses-1` IS NULL THEN 0 ELSE `MaxBucketExpenses-1` END) THEN 
                        (CASE WHEN `MaxBucketInterests-1` IS NULL THEN 0 ELSE `MaxBucketInterests-1` END)
			   ELSE (CASE WHEN `MaxBucketCapital-1` IS NULL THEN 0 ELSE `MaxBucketCapital-1` END) 
               END AS MAX_1
		FROM Max3
'''
)
Max4.createOrReplaceTempView("Max4")

In [0]:
Max5 = spark.sql(      
f'''
		SELECT *
		     , CASE WHEN (CASE WHEN `MaxBucketCapital` IS NULL THEN 0 ELSE `MaxBucketCapital` END) = 
                           (CASE WHEN MAX IS NULL THEN 0 ELSE MAX END) THEN 
                           (CASE WHEN MaxBucketCapitalAmount IS NULL THEN 0 ELSE MaxBucketCapitalAmount END) +
                           (CASE WHEN TotalExpenses IS NULL THEN 0 ELSE TotalExpenses END) +
                           (CASE WHEN TotalInterestsAndMLT IS NULL THEN 0 ELSE TotalInterestsAndMLT END)
			        WHEN (CASE WHEN MaxBucketInterests IS NULL THEN 0 ELSE MaxBucketInterests END) >= 
                           (CASE WHEN MaxBucketExpenses IS NULL THEN 0 ELSE MaxBucketExpenses END) THEN 
                           (CASE WHEN TotalExpenses IS NULL THEN 0 ELSE TotalExpenses END) +
                           (CASE WHEN MaxBucketInterestsAndMLT IS NULL THEN 0 ELSE MaxBucketInterestsAndMLT END)
		       ELSE (CASE WHEN MaxBucketExpensesAmount IS NULL THEN 0 ELSE MaxBucketExpensesAmount END) 
               END AS MinimumPayoutAmount
			 , CASE WHEN (CASE WHEN MAX IS NULL THEN 0 ELSE MAX END) - 
                            (CASE WHEN MAX_1 IS NULL THEN 0 ELSE MAX_1 END) > 1 THEN 0
			        WHEN (CASE WHEN MaxBucketCapital IS NULL THEN 0 ELSE MaxBucketCapital END) = 
                            (CASE WHEN MAX IS NULL THEN 0 ELSE MAX END) AND 
                            (CASE WHEN MAX_1 IS NULL THEN 0 ELSE MAX_1 END) = 
                            (CASE WHEN `MaxBucketInterests-1` IS NULL THEN 0 ELSE `MaxBucketInterests-1` END) THEN 0
					WHEN (CASE WHEN MaxBucketCapital IS NULL THEN 0 ELSE MaxBucketCapital END) = 
                            (CASE WHEN MAX IS NULL THEN 0 ELSE MAX END) AND 
                            (CASE WHEN MAX_1 IS NULL THEN 0 ELSE MAX_1 END) = 
                            (CASE WHEN `MaxBucketExpenses-1` IS NULL THEN 0 ELSE `MaxBucketExpenses-1` END) THEN 0
					WHEN (CASE WHEN MaxBucketInterests IS NULL THEN 0 ELSE MaxBucketInterests END) = 
                            (CASE WHEN MAX IS NULL THEN 0 ELSE MAX END) AND 
                            (CASE WHEN MAX_1 IS NULL THEN 0 ELSE MAX_1 END) = 
                            (CASE WHEN `MaxBucketExpenses-1` IS NULL THEN 0 ELSE `MaxBucketExpenses-1` END) THEN 0
					WHEN (CASE WHEN MaxBucketCapital IS NULL THEN 0 ELSE MaxBucketCapital END) =  
                            (CASE WHEN MAX IS NULL THEN 0 ELSE MAX END) AND 
                            (CASE WHEN MAX_1 IS NULL THEN 0 ELSE MAX_1 END) = 
                            (CASE WHEN `MaxBucketCapital-1` IS NULL THEN 0 ELSE `MaxBucketCapital-1` END) THEN 
                            (CASE WHEN `MaxBucketCapitalAmount-1` IS NULL THEN 0 ELSE `MaxBucketCapitalAmount-1` END)
		            WHEN (CASE WHEN MAX IS NULL THEN 0 ELSE MAX END) = 
                            (CASE WHEN MaxBucketInterests IS NULL THEN 0 ELSE MaxBucketInterests END) AND 
                            (CASE WHEN MAX_1 IS NULL THEN 0 ELSE MAX_1 END) = 
                            (CASE WHEN `MaxBucketCapital-1` IS NULL THEN 0 ELSE `MaxBucketCapital-1` END) THEN 
                         (CASE WHEN TotalInterests IS NULL THEN 0 ELSE TotalInterests END) - 
                         (CASE WHEN MaxBucketInterests IS NULL THEN 0 ELSE MaxBucketInterests END) + 
                         (CASE WHEN MaxBucketCapitalAmount IS NULL THEN 0 ELSE MaxBucketCapitalAmount END)
					WHEN (CASE WHEN MAX IS NULL THEN 0 ELSE MAX END) = 
                         (CASE WHEN MaxBucketExpenses IS NULL THEN 0 ELSE MaxBucketExpenses END) AND 
                         (CASE WHEN MAX_1 IS NULL THEN 0 ELSE MAX_1 END) = 
                         (CASE WHEN `MaxBucketCapital-1` IS NULL THEN 0 ELSE `MaxBucketCapital-1` END) THEN 
                         (CASE WHEN TotalExpenses IS NULL THEN TotalExpenses ELSE TotalExpenses END) - 
                         (CASE WHEN MaxBucketExpensesAmount IS NULL THEN 0 ELSE MaxBucketExpensesAmount END) + 
                         (CASE WHEN TotalInterestsAndMLT IS NULL THEN 0 ELSE TotalInterestsAndMLT END) +
                         (CASE WHEN MaxBucketCapitalAmount IS NULL THEN 0 ELSE MaxBucketCapitalAmount END)
					WHEN (CASE WHEN MAX IS NULL THEN 0 ELSE MAX END) = 
                         (CASE WHEN MaxBucketExpenses IS NULL THEN 0 ELSE MaxBucketExpenses END) AND 
                         (CASE WHEN MAX_1 IS NULL THEN 0 ELSE MAX_1 END) = 
                         (CASE WHEN `MaxBucketInterests-1` IS NULL THEN 0 ELSE `MaxBucketInterests-1` END) THEN 
                         (CASE WHEN TotalExpenses IS NULL THEN 0 ELSE TotalExpenses END) - 
                         (CASE WHEN MaxBucketExpensesAmount IS NULL THEN 0 ELSE MaxBucketExpensesAmount END) + 
                         (CASE WHEN MaxBucketInterests IS NULL THEN 0 ELSE MaxBucketInterests END) +
                         (CASE WHEN `DueInterest` IS NULL THEN 0 ELSE `DueInterest` END)
					WHEN (CASE WHEN MAX IS NULL THEN 0 ELSE MAX END) = 
                         (CASE WHEN MaxBucketExpenses IS NULL THEN 0 ELSE MaxBucketExpenses END) AND 
                         (CASE WHEN MAX_1 IS NULL THEN 0 ELSE MAX_1 END) = 
                         (CASE WHEN `MaxBucketExpenses-1` IS NULL THEN 0 ELSE `MaxBucketExpenses-1` END) THEN 
                         (CASE WHEN `MaxBucketExpensesAmount-1` IS NULL THEN 0 ELSE `MaxBucketExpensesAmount-1` END)
				    WHEN (CASE WHEN MAX IS NULL THEN 0 ELSE MAX END) = 
                         (CASE WHEN MaxBucketInterests IS NULL THEN 0 ELSE MaxBucketInterests END) AND 
                         (CASE WHEN MAX_1 IS NULL THEN 0 ELSE MAX_1 END) = 
                         (CASE WHEN `MaxBucketInterests-1` IS NULL THEN 0 ELSE `MaxBucketInterests-1` END) THEN 
                         (CASE WHEN `MaxBucketInterests-1` IS NULL THEN 0 ELSE `MaxBucketInterests-1` END)
			   ELSE '9999999999' 
               END AS `MinimumPayoutAmountb-1`
		FROM Max4
'''
)
Max5.createOrReplaceTempView("Max5")

In [0]:
AmountFor90 = spark.sql(      
f'''
		SELECT *
		     , CASE WHEN (CASE WHEN `Capital>2` IS NULL THEN 0 ELSE `Capital>2` END) > 0 THEN 
                           (CASE WHEN `Capital>2` IS NULL THEN 0 ELSE `Capital>2` END) +
                           (CASE WHEN `MLTandInterests>2` IS NULL THEN 0 ELSE `MLTandInterests>2` END) + 
                           (CASE WHEN `Interests1-2` IS NULL THEN 0 ELSE `Interests1-2` END) +
                           (CASE WHEN `Expenses1-2` IS NULL THEN 0 ELSE `Expenses1-2` END) + 
                           (CASE WHEN `Expenses>2` IS NULL THEN 0 ELSE `Expenses>2` END)
			       WHEN (CASE WHEN `MLTandInterests>2` IS NULL THEN 0 ELSE `MLTandInterests>2` END) > 0 THEN 
                           (CASE WHEN `MLTandInterests>2` IS NULL THEN 0 ELSE `MLTandInterests>2` END) + 
                           (CASE WHEN `Expenses1-2` IS NULL THEN 0 ELSE `Expenses1-2` END) +
                           (CASE WHEN `Expenses>2` IS NULL THEN 0 ELSE `Expenses>2` END)
				  WHEN (CASE WHEN `Expenses>2` IS NULL THEN 0 ELSE `Expenses>2` END) > 0 THEN 
                           (CASE WHEN `Expenses>2` IS NULL THEN 0 ELSE `Expenses>2` END)
			   ELSE 0 
               END AS `AmountFor90-`
		FROM MAX3
'''
)
AmountFor90.createOrReplaceTempView("AmountFor90")

##### InflowMinAmounts

In [0]:
InflowMinAmounts_v1 = spark.sql(      
f'''
		SELECT  *
			,   CAST(CASE WHEN (CASE WHEN `Rule100` IS NULL THEN 0 ELSE `Rule100` END) > 
                          (CASE WHEN `Rule1%` IS NULL THEN 0 ELSE `Rule1%` END) THEN 
                          (CASE WHEN `Rule1%` IS NULL THEN 0 ELSE `Rule1%` END)
			         WHEN (CASE WHEN `Rule100` IS NULL THEN 0 ELSE `Rule100` END) < 
                          (CASE WHEN `Rule1%` IS NULL THEN 0 ELSE `Rule1%` END) THEN 
                          (CASE WHEN `Rule100` IS NULL THEN 0 ELSE `Rule100` END)
					     WHEN (CASE WHEN `Rule100` IS NULL THEN 0 ELSE `Rule100` END) = 
                          (CASE WHEN `Rule1%` IS NULL THEN 0 ELSE `Rule1%` END) THEN 
                          (CASE WHEN `Rule1%` IS NULL THEN 0 ELSE `Rule1%` END) END AS DECIMAL(19,2)) AS MIN      
		FROM AccountPerimeter_V2
'''
)
InflowMinAmounts_v1.createOrReplaceTempView("InflowMinAmounts_v1")

In [0]:
InflowMinAmounts_v2 = spark.sql(      
f'''
		SELECT DISTINCT
				f.*
			,   s.SavedReason
			,   s.FinalRiskSaved
			,	s.SavedFlag AS FinalSaved
			,   m6.`MinimumPayoutAmountb-1`
			,   m6.MinimumPayoutAmount
			,   CASE WHEN f.MIN > 0 THEN f.MIN
			    ELSE 0 END AS MinAmount
			,   m.`AmountFor90-`         
		FROM InflowMinAmounts_v1 f
		LEFT JOIN AmountFor90  m
		ON m.`ExposureKeySIGLO` = f.ExposureKey
		LEFT JOIN Saved s
		ON s.Account = f.ExposureKey
		LEFT JOIN Max5        m6
	    ON m6.`ExposureKeySIGLO` = f.ExposureKey
'''
)
InflowMinAmounts_v2.createOrReplaceTempView("InflowMinAmounts_v2")

In [0]:
InflowMinAmounts_v3 = spark.sql(      
f'''
		SELECT 
			    ExposureKey			
		    ,	AssetSegment				
		    ,	CDI
		    ,   SavedReason
		    ,   FinalRiskSaved
		    ,	FinalSaved			
		    ,	Bucket	
            ,   AssignmentChannel
            ,   AssignmentTeam as AssignmentChannelDetailed 
            ,   AssignmentSegmentDetailed 
            ,   AccountSegment 
            ,   EBAStatus as EBACategory 
            ,   CourseOfAction as COA 
			,   (CASE WHEN `MinimumPayoutAmountb-1` IS NULL THEN 0 ELSE `MinimumPayoutAmountb-1` END)   AS `MinimumPayoutAmountb-1`
			,   (CASE WHEN `MinimumPayoutAmount` IS NULL THEN 0 ELSE `MinimumPayoutAmount` END)         AS `MinimumPayoutAmount`
			,   (CASE WHEN `AmountFor90-` IS NULL THEN 0 ELSE `AmountFor90-` END)                           AS `AmountFor90-`
			,   (CASE WHEN MinAmount IS NULL THEN 0 ELSE MinAmount END)                                     AS `MinAmount1+`
		    ,   CASE WHEN Bucket < 4 AND AssetSegment = 'SB' AND `RiskBucket` = 0 THEN 
                            (CASE WHEN CAST(`MinimumPayoutAmount` AS DECIMAL(19,2)) IS NULL THEN 0 ELSE CAST(`MinimumPayoutAmount` AS DECIMAL(19,2)) END)
			         WHEN AssetSegment = 'SB' AND `RiskBucket` = 0 THEN 
                            (CASE WHEN CAST(`AmountFor90-` AS DECIMAL(19,2)) IS NULL THEN 0 ELSE CAST(`AmountFor90-` AS DECIMAL(19,2)) END)
		        ELSE (CASE WHEN CAST(MinAmount AS DECIMAL(19,2)) IS NULL THEN 0 ELSE CAST(MinAmount AS DECIMAL(19,2)) END) END AS MinAmount
			,   CASE WHEN AssetSegment = 'SB' AND `RiskBucket` = 0 THEN 'SB Bucket 0' ELSE 'Bucket 1+' END AS Flag
		FROM InflowMinAmounts_v2
'''
)
InflowMinAmounts_v3.createOrReplaceTempView("InflowMinAmounts_v3")

In [0]:
InflowMinAmounts = spark.sql(      
f'''
		SELECT 
			    ExposureKey			
		    ,	AssetSegment				
		    ,	CDI
		    ,   SavedReason
		    ,   FinalRiskSaved
		    ,	FinalSaved			
		    ,	Bucket	
            ,   AssignmentChannel
            ,   AssignmentChannelDetailed 
            ,   AssignmentSegmentDetailed 
            ,   AccountSegment 
            ,   EBACategory 
            ,   COA 
			,   `MinimumPayoutAmountb-1`
			,   MinimumPayoutAmount
			,   `AmountFor90-`
			,   `MinAmount1+`
		    ,   CASE WHEN FinalSaved = 'Saved' THEN 0
		             ELSE MinAmount END AS MinAmount
		    ,   Flag
		FROM InflowMinAmounts_v3
'''
)
InflowMinAmounts.createOrReplaceTempView("InflowMinAmounts")

##### fina_df

In [0]:
final_df = spark.sql(      
f'''
		SELECT    
              CAST({run_date!r} AS DATE)                          AS BusinessDate
            , CAST(ExposureKey AS STRING)                         AS ExposureKey
            , CAST(AssetSegment AS STRING)                        AS AssetSegment
            , CAST(CDI AS STRING)                                 AS CDI
            , CAST(SavedReason  AS STRING)                        AS SavedReason
            , CAST(FinalRiskSaved AS STRING)                      AS FinalRiskSaved
            , CAST(FinalSaved AS STRING)                          AS FinalSaved
            , CAST(Bucket AS INT)                                 AS Bucket
            , CAST(AssignmentChannel AS STRING)                   AS AssignmentChannel
            , CAST(AssignmentChannelDetailed AS STRING)           AS AssignmentChannelDetailed
            , CAST(AssignmentSegmentDetailed AS STRING)           AS AssignmentSegmentDetailed
            , CAST(AccountSegment AS STRING)                      AS AccountSegment 
            , CAST(EBACategory  AS STRING)                        AS EBACategory
            , CAST(COA AS STRING)                                 AS COA
            , CAST(`MinimumPayoutAmountb-1` AS DECIMAL(19,2))   AS `MinimumPayoutAmountb-1`
		, CAST(MinimumPayoutAmount AS DECIMAL(19,2))        AS MinimumPayoutAmount
            , CAST(`AmountFor90-` AS DECIMAL(19,2))               AS `AmountFor90-`
            , CAST(`MinAmount1+` AS DECIMAL(19,2))                AS `MinAmount1+`
            , CAST(MinAmount AS DECIMAL(19,2))                    AS MinAmount
            , CAST(Flag AS STRING)                                AS Flag
            , CAST(getdate() AS TIMESTAMP)                        AS InsertTimeStamp
		FROM InflowMinAmounts
'''
)
final_df.createOrReplaceTempView("final_df")

In [0]:
if final_df.rdd.isEmpty():
    raise ValueError(f"Core_InflowMinAmounts: final DataFrame is empty - aborting write to {delivery_table_path} to avoid overwriting with no data")
final_df.write \
    .format("delta") \
    .partitionBy("BusinessDate") \
    .mode("append") \
    .saveAsTable(delivery_table_path) # in order to write in delta_lake

In [0]:
%python
from pyspark.sql.functions import col, count, concat

def check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="error"):
    errors = []
    warnings = []

    if key_columns:
        # Create the composite key column if there are multiple key columns
        if len(key_columns) > 1:
            composite_key = concat(*[col(c) for c in key_columns]).alias("CompositeKey")
            key_column_name = "CompositeKey"
            key_columns_string = ", ".join(key_columns)
        else:
            key_column_name = key_columns[0]
            key_columns_string = key_columns[0]

        # Duplicate Checks
        duplicate_count = (
            spark.table(delivery_table_path)
            .filter(col("BusinessDate") == run_date)
            .withColumn("CompositeKey", composite_key if len(key_columns) > 1 else col(key_column_name))
            .groupBy(key_column_name)
            .count()
            .filter("count > 1")
            .select(key_column_name, "count")
            .toPandas()
            .to_dict(orient='records')
        )

        if duplicate_count:
            message = f"Duplicate {key_columns_string} entries found: {duplicate_count}"
            if duplicate_check_action == "error":
                errors.append(message)
            elif duplicate_check_action == "warning":
                warnings.append(message)

    # Check if the table for the specific date is empty
    row_count = spark.sql(f"""
        SELECT * FROM {delivery_table_path}
        WHERE BusinessDate = '{run_date}'
    """).count()

    if row_count == 0:
        message = f"The {delivery_table_path} dataset is empty for the date {run_date}."
        if empty_check_action == "error":
            errors.append(message)
        elif empty_check_action == "warning":
            warnings.append(message)

    if warnings:
        print("Warnings:\n" + "\n".join(warnings))

    if errors:
        raise ValueError("Errors found:\n" + "\n".join(errors))

key_columns = ["ExposureKey"]
check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="warning")

In [0]:
import re
def rename_and_cleanup_files(output_path, file_name):
    part_file_regex = re.compile(r'part-.*')
    system_file_regex = re.compile(r'^_(committed_|started_|SUCCESS)')
 
    # Move the part file to the final filename.
    files = dbutils.fs.ls(output_path)
    for file in files:
        print(f"file: {file.name}")
        if part_file_regex.match(file.name):
            original_path = file.path
            new_path = output_path + file_name
            dbutils.fs.mv(original_path, new_path)
 
    # Remove system files from the common folder.
    files = dbutils.fs.ls(output_path)
    for file in files:
        if system_file_regex.match(file.name):
            dbutils.fs.rm(file.path, recurse=False)
 
# Example usage:
# rename_and_cleanup_files(output_path, file_name)

In [0]:
InflowMinAmounts = spark.sql(
f'''
select * from {delivery_table_path}
where BusinessDate = {run_date!r}
'''
)
InflowMinAmounts.createOrReplaceTempView("InflowMinAmounts")

In [0]:
output_path = "/Volumes/"+env_prefix+"-cdh-delivery/repoint/exports/Core/InflowMinAmounts/"
file_name = "InflowMinAmounts_" +run_date+ ".csv"
InflowMinAmounts.coalesce(1).write.mode("append").option("header", True).csv(f"{output_path}")
rename_and_cleanup_files(output_path, file_name)